# Item response distributions per sample

Documents the justification for ordinal (WLSMV) estimation given in the preregistration
(`prereg.txt`, invariance section): *"All models will use the WLSMV estimator since the scale
items only have 4 response levels and preliminary examination of the HiTOP validation sample
indicates non-normal response distributions."* No ordinal-vs-continuous model comparison was
run; this notebook quantifies the two premises (number of categories, non-normality) for every
item that enters an invariance ladder, in every sample that enters one.

Descriptive only (exploratory) — no models are fit. Benchmarks used below:

- **Number of categories.** Simulation work recommends categorical estimation below five
  response categories (Rhemtulla, Brosseau-Liard & Savalei, 2012, *Psychological Methods*);
  every item here has four.
- **Non-normality.** |skew| > 1 is reported as moderate and |skew| > 2 / excess kurtosis > 7 as
  the conventional thresholds for substantial non-normality under ML (Curran, West & Finch,
  1996).
- **Sparse categories.** Categories with 0 or < 10 responses in a sample, which make thresholds
  poorly determined in the multi-group ordinal models.

Outputs (`data/item_distributions/`): `item_distributions.csv` (one row per item × sample),
`sample_summary.csv` (Table 1), `scale_summary.csv` (Table 2), `floor_gaps.csv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

from hitop_cfa.scales import ORIG_ITEMS, OTHER_SCALES

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 250)

dat_dir = Path('../data'); fin_dir = dat_dir / 'finaldata'
out_dir = dat_dir / 'item_distributions'; out_dir.mkdir(exist_ok=True)

## Items and response coding

Item sets come from `hitop_cfa.scales` (the same formulas the ladders use). All instruments have
four ordered response options: HiTOP items are coded 1–4, PHQ-8, GAD-7 and BAARS-IV items 0–3.
Categories are reported positionally (`cat1` = lowest, e.g. HiTOP "Not at all").

In [2]:
def formula_items(formula):
    return formula.split('=~', 1)[1].strip().split(' + ')

def instrument_of(scale):
    if scale in ORIG_ITEMS:
        return 'HiTOP'
    return {'phq_sum': 'PHQ-8', 'gad_sum': 'GAD-7'}.get(scale, 'BAARS-IV')

item_info = pd.DataFrame(
    [dict(item=i, scale=s, instrument=instrument_of(s))
     for s, f in list(ORIG_ITEMS.items()) + list(OTHER_SCALES.items())
     for i in formula_items(f)])
assert not item_info.item.duplicated().any()
CATS = {'HiTOP': [1, 2, 3, 4], 'PHQ-8': [0, 1, 2, 3], 'GAD-7': [0, 1, 2, 3], 'BAARS-IV': [0, 1, 2, 3]}
INSTRUMENTS = list(CATS)
item_info.groupby('instrument', sort=False).size()

instrument
HiTOP       83
PHQ-8        8
GAD-7        7
BAARS-IV    27
dtype: int64

## Samples

The five groups that enter an invariance ladder:

| sample | file | used in |
|---|---|---|
| Validation | `dat_val.csv` (HiTOP items only) | V_GP, V_EN |
| GP wave 1 | `dat_gp_grid1st_full.csv` | V_GP, GP_EN, other-scale GP_EN |
| EN wave 1 | `dat_en_grid1st_full.csv` | V_EN, GP_EN, other-scale GP_EN |
| GP wave 2 | `dat_gp_gridall_full.csv`, `*_recontact` columns | wave and crossed-wave ladders |
| EN wave 2 | `dat_en_gridall_full.csv`, `*_recontact` columns | wave and crossed-wave ladders |

The wave ladders pair wave 2 with the wave-1 responses of the recontact cohort, which is a
subset of the full wave-1 samples above and is not tabulated separately. Wave-2 frames rename
the `*_recontact` columns to plain item ids, as in the wave notebooks.

In [3]:
def wave_frame(df, suffix):
    cols = [c for c in item_info.item if c + suffix in df.columns]
    out = df[[c + suffix for c in cols]].copy()
    out.columns = cols
    return out

data_val = pd.read_csv(fin_dir / 'dat_val.csv')
data_gp = pd.read_csv(fin_dir / 'dat_gp_grid1st_full.csv', low_memory=False)
data_en = pd.read_csv(fin_dir / 'dat_en_grid1st_full.csv', low_memory=False)
data_gp_all = pd.read_csv(fin_dir / 'dat_gp_gridall_full.csv', low_memory=False)
data_en_all = pd.read_csv(fin_dir / 'dat_en_gridall_full.csv', low_memory=False)

samples = {
    'Validation': wave_frame(data_val, ''),
    'GP wave 1': wave_frame(data_gp, ''),
    'EN wave 1': wave_frame(data_en, ''),
    'GP wave 2': wave_frame(data_gp_all, '_recontact'),
    'EN wave 2': wave_frame(data_en_all, '_recontact'),
}
SAMPLES = list(samples)

# every item present must be complete and on its instrument's 4-point coding
for name, df in samples.items():
    assert not df.isna().any().any(), name
    for inst, cats in CATS.items():
        cols = [c for c in item_info.loc[item_info.instrument == inst, 'item'] if c in df]
        assert set(np.unique(df[cols].values)) <= set(cats), (name, inst)
pd.DataFrame({name: dict(n=len(df), items=df.shape[1]) for name, df in samples.items()}).T

,n,items
Validation,496,83
GP wave 1,497,125
EN wave 1,305,125
GP wave 2,398,125
EN wave 2,255,125


## Item-level distributions

Per item × sample: category counts and proportions, the share in the lowest and highest
category, the number of empty categories, the smallest category count, and bias-corrected skew
and excess kurtosis (`scipy.stats`, `bias=False`).

In [4]:
rows = []
for sample, df in samples.items():
    for info in item_info.itertuples():
        if info.item not in df:
            continue
        cats = CATS[info.instrument]
        x = df[info.item]
        counts = x.value_counts().reindex(cats, fill_value=0).to_numpy()
        props = counts / counts.sum()
        row = dict(sample=sample, instrument=info.instrument, scale=info.scale, item=info.item,
                   n=int(counts.sum()))
        row.update({f'n_cat{k + 1}': int(c) for k, c in enumerate(counts)})
        row.update({f'p_cat{k + 1}': p for k, p in enumerate(props)})
        row.update(p_lowest=props[0], p_highest=props[-1],
                   n_empty_cats=int((counts == 0).sum()), min_cat_n=int(counts.min()),
                   skew=stats.skew(x, bias=False), kurtosis=stats.kurtosis(x, bias=False))
        rows.append(row)
items = pd.DataFrame(rows)
items.to_csv(out_dir / 'item_distributions.csv', index=False)
print(f'{len(items)} item × sample rows')
items.head()

583 item × sample rows


,sample,instrument,scale,item,n,n_cat1,n_cat2,n_cat3,n_cat4,p_cat1,p_cat2,p_cat3,p_cat4,p_lowest,p_highest,n_empty_cats,min_cat_n,skew,kurtosis
0,Validation,HiTOP,anhedonic_depression,hitop39,496,73,133,168,122,0.147177,0.268145,0.338710,0.245968,0.147177,0.245968,0,73,-0.215593,-1.025667
1,Validation,HiTOP,anhedonic_depression,hitop77,496,164,164,103,65,0.330645,0.330645,0.207661,0.131048,0.330645,0.131048,0,65,0.458831,-0.936623
2,Validation,HiTOP,anhedonic_depression,hitop84,496,56,119,162,159,0.112903,0.239919,0.326613,0.320565,0.112903,0.320565,0,56,-0.394422,-0.944810
3,Validation,HiTOP,anhedonic_depression,hitop92,496,81,146,137,132,0.163306,0.294355,0.276210,0.266129,0.163306,0.266129,0,81,-0.116717,-1.184277
4,Validation,HiTOP,anhedonic_depression,hitop93,496,103,141,140,112,0.207661,0.284274,0.282258,0.225806,0.207661,0.225806,0,103,-0.022793,-1.211966


## Table 1 — per-sample summary

One row per sample × instrument. Ranges are across the instrument's items in that sample.

In [5]:
def summarise(g):
    return pd.Series(dict(
        n=g.n.iloc[0],
        n_items=len(g),
        pct_lowest_median=100 * g.p_lowest.median(),
        pct_lowest_min=100 * g.p_lowest.min(),
        pct_lowest_max=100 * g.p_lowest.max(),
        items_lowest_ge50=int((g.p_lowest >= .5).sum()),
        pct_highest_median=100 * g.p_highest.median(),
        skew_median=g['skew'].median(),
        skew_min=g['skew'].min(),
        skew_max=g['skew'].max(),
        items_abs_skew_gt1=int((g['skew'].abs() > 1).sum()),
        items_abs_skew_gt2=int((g['skew'].abs() > 2).sum()),
        kurtosis_median=g['kurtosis'].median(),
        kurtosis_min=g['kurtosis'].min(),
        kurtosis_max=g['kurtosis'].max(),
        items_kurtosis_gt7=int((g['kurtosis'] > 7).sum()),
        items_empty_cat=int((g.n_empty_cats > 0).sum()),
        items_cat_lt10=int((g.min_cat_n < 10).sum()),
    ))

summary = (items.groupby(['sample', 'instrument'], sort=False)
           .apply(summarise, include_groups=False).reset_index())
summary['sample'] = pd.Categorical(summary['sample'], SAMPLES, ordered=True)
summary['instrument'] = pd.Categorical(summary['instrument'], INSTRUMENTS, ordered=True)
summary = summary.sort_values(['instrument', 'sample']).reset_index(drop=True)
summary.to_csv(out_dir / 'sample_summary.csv', index=False)

def frac(k, n):
    return f'{int(k)}/{int(n)}'

table1 = pd.DataFrame({
    'Instrument': summary.instrument.astype(str),
    'Sample': summary['sample'].astype(str),
    'N': summary.n.astype(int),
    'Items': summary.n_items.astype(int),
    '% lowest cat., median [range]': [
        f'{r.pct_lowest_median:.0f} [{r.pct_lowest_min:.0f}–{r.pct_lowest_max:.0f}]'
        for r in summary.itertuples()],
    'Items ≥50% lowest': [frac(r.items_lowest_ge50, r.n_items) for r in summary.itertuples()],
    '% highest cat., median': summary.pct_highest_median.round(0).astype(int),
    'Skew, median [range]': [
        f'{r.skew_median:.2f} [{r.skew_min:.2f}–{r.skew_max:.2f}]' for r in summary.itertuples()],
    '|skew| > 1': [frac(r.items_abs_skew_gt1, r.n_items) for r in summary.itertuples()],
    '|skew| > 2': [frac(r.items_abs_skew_gt2, r.n_items) for r in summary.itertuples()],
    'Excess kurtosis, median [range]': [
        f'{r.kurtosis_median:.2f} [{r.kurtosis_min:.2f}–{r.kurtosis_max:.2f}]'
        for r in summary.itertuples()],
    'Kurtosis > 7': [frac(r.items_kurtosis_gt7, r.n_items) for r in summary.itertuples()],
    'Empty category': [frac(r.items_empty_cat, r.n_items) for r in summary.itertuples()],
    'Category n < 10': [frac(r.items_cat_lt10, r.n_items) for r in summary.itertuples()],
})
table1

,Instrument,Sample,N,Items,"% lowest cat., median [range]",Items ≥50% lowest,"% highest cat., median","Skew, median [range]",|skew| > 1,|skew| > 2,"Excess kurtosis, median [range]",Kurtosis > 7,Empty category,Category n < 10
0,HiTOP,Validation,496,83,25 [5–77],11/83,17,0.23 [-0.64–2.14],10/83,1/83,-0.98 [-1.37–3.77],0/83,0/83,0/83
1,HiTOP,GP wave 1,497,83,55 [13–89],50/83,8,1.11 [-0.28–3.55],49/83,10/83,0.13 [-1.26–12.47],1/83,0/83,1/83
2,HiTOP,EN wave 1,305,83,28 [3–79],12/83,22,0.20 [-0.71–2.15],10/83,1/83,-1.01 [-1.46–3.51],0/83,0/83,2/83
3,HiTOP,GP wave 2,398,83,58 [10–88],57/83,8,1.22 [-0.12–3.10],56/83,12/83,0.55 [-1.20–8.68],2/83,0/83,2/83
4,HiTOP,EN wave 2,255,83,25 [4–81],13/83,22,0.16 [-0.69–2.32],12/83,2/83,-0.99 [-1.43–4.57],0/83,0/83,6/83
5,PHQ-8,GP wave 1,497,8,54 [34–82],6/8,9,1.16 [0.58–2.67],6/8,1/8,0.25 [-0.91–7.01],1/8,0/8,0/8
6,PHQ-8,EN wave 1,305,8,13 [7–50],1/8,32,-0.15 [-0.60–1.05],1/8,0/8,-1.11 [-1.48–0.11],0/8,0/8,0/8
7,PHQ-8,GP wave 2,398,8,54 [37–81],6/8,8,1.12 [0.67–2.38],6/8,1/8,0.23 [-0.68–4.90],0/8,0/8,1/8
8,PHQ-8,EN wave 2,255,8,12 [5–55],1/8,32,-0.15 [-0.66–1.15],1/8,0/8,-1.09 [-1.37–0.37],0/8,0/8,0/8
9,GAD-7,GP wave 1,497,7,50 [43–68],4/7,10,1.04 [0.89–1.71],5/7,0/7,0.05 [-0.22–2.06],0/7,0/7,0/7


Pooled across samples (each item × sample counted once):

In [6]:
pooled = items.groupby('instrument', sort=False).agg(
    item_x_sample=('item', 'size'),
    pct_lowest_median=('p_lowest', lambda s: 100 * s.median()),
    share_abs_skew_gt1=('skew', lambda s: (s.abs() > 1).mean()),
    share_abs_skew_gt2=('skew', lambda s: (s.abs() > 2).mean()),
    share_kurtosis_gt7=('kurtosis', lambda s: (s > 7).mean()),
    share_cat_lt10=('min_cat_n', lambda s: (s < 10).mean()),
).round(2)
pooled

,item_x_sample,pct_lowest_median,share_abs_skew_gt1,share_abs_skew_gt2,share_kurtosis_gt7,share_cat_lt10
instrument,,,,,,
HiTOP,415,37.44,0.33,0.06,0.01,0.03
PHQ-8,32,39.32,0.44,0.06,0.03,0.03
GAD-7,28,37.76,0.36,0.00,0.00,0.00
BAARS-IV,108,43.26,0.42,0.02,0.02,0.06


## Table 2 — per-scale summary

Median % of responses in the lowest category across a scale's items, and the median item skew,
for each sample (blank where the instrument was not administered).

In [7]:
scale_order = list(ORIG_ITEMS) + list(OTHER_SCALES)
by_scale = (items.groupby(['instrument', 'scale', 'sample'], sort=False)
            .agg(n_items=('item', 'size'),
                 pct_lowest_median=('p_lowest', lambda s: 100 * s.median()),
                 skew_median=('skew', 'median'),
                 skew_max_abs=('skew', lambda s: s.abs().max()),
                 items_cat_lt10=('min_cat_n', lambda s: int((s < 10).sum())))
            .reset_index())
by_scale.to_csv(out_dir / 'scale_summary.csv', index=False)

def scale_pivot(value, fmt):
    p = by_scale.pivot(index='scale', columns='sample', values=value)
    return p.reindex(index=scale_order, columns=SAMPLES).map(
        lambda v: '' if pd.isna(v) else fmt.format(v))

print('Median % in lowest category')
display(scale_pivot('pct_lowest_median', '{:.0f}'))
print('Median item skew')
display(scale_pivot('skew_median', '{:.2f}'))

Median % in lowest category


sample,Validation,GP wave 1,EN wave 1,GP wave 2,EN wave 2
scale,,,,,
anhedonic_depression,20,51,16,54,15
anxious_worry,15,46,9,51,8
appetite_gain,38,49,35,54,33
appetite_loss,47,64,40,69,40
cognitive_problems,19,47,9,49,10
hyposomnia,57,74,66,76,69
indecisiveness,16,53,12,55,11
insomnia,20,39,18,37,16
panic,39,69,33,68,33


Median item skew


sample,Validation,GP wave 1,EN wave 1,GP wave 2,EN wave 2
scale,,,,,
anhedonic_depression,0.04,1.09,-0.06,1.17,-0.12
anxious_worry,-0.28,0.78,-0.32,0.92,-0.55
appetite_gain,0.59,1.02,0.46,1.24,0.42
appetite_loss,0.79,1.55,0.64,1.59,0.71
cognitive_problems,0.01,0.89,-0.08,1.04,-0.14
hyposomnia,1.19,2.10,1.46,2.15,1.64
indecisiveness,0.00,1.05,-0.03,1.18,-0.02
insomnia,0.01,0.55,-0.25,0.59,-0.26
panic,0.69,1.70,0.57,1.69,0.53


## Floor differences between compared groups

For each cross-sample pair, the per-item difference in the share of lowest-category responses
(second group minus first, percentage points). Wave pairs are omitted because their wave-1 group
is the recontact cohort, not the full wave-1 sample tabulated here.

In [8]:
PAIRS = [('V_GP', 'HiTOP', 'Validation', 'GP wave 1'),
         ('V_EN', 'HiTOP', 'Validation', 'EN wave 1'),
         ('GP_EN', 'HiTOP', 'GP wave 1', 'EN wave 1'),
         ('GP_EN (PHQ/GAD/BAARS)', 'other', 'GP wave 1', 'EN wave 1')]
gap_rows = []
for pair, inst, a, b in PAIRS:
    sel = items[items.instrument.eq('HiTOP') if inst == 'HiTOP' else items.instrument.ne('HiTOP')]
    w = sel.pivot(index='item', columns='sample', values='p_lowest')[[a, b]].dropna()
    gap = 100 * (w[b] - w[a])
    gap_rows.append(dict(pair=pair, groups=f'{b} − {a}', n_items=len(gap),
                         gap_pp_median=gap.median(), gap_pp_min=gap.min(), gap_pp_max=gap.max(),
                         items_abs_gap_ge20=int((gap.abs() >= 20).sum())))
floor_gaps = pd.DataFrame(gap_rows).round(1)
floor_gaps.to_csv(out_dir / 'floor_gaps.csv', index=False)
floor_gaps

,pair,groups,n_items,gap_pp_median,gap_pp_min,gap_pp_max,items_abs_gap_ge20
0,V_GP,GP wave 1 − Validation,83,27.9,-12.7,41.6,54
1,V_EN,EN wave 1 − Validation,83,-1.4,-13.0,27.4,5
2,GP_EN,EN wave 1 − GP wave 1,83,-29.0,-42.1,27.7,62
3,GP_EN (PHQ/GAD/BAARS),EN wave 1 − GP wave 1,42,-32.7,-47.2,-20.0,42


## Reading

1. **Four categories everywhere, none empty.** Every item has four ordered categories, all
   observed in every sample. 19 of 583 item × sample cells have a category with fewer than 10
   responses, always the highest category, mostly in the GP samples and EN wave 2.
2. **Shape depends on the sample.** The GP samples (both waves) are floor-heavy and right-skewed:
   at wave 1 a median 55% of HiTOP responses are in the lowest category, 49/83 items have
   |skew| > 1 and 10/83 have |skew| > 2; PHQ-8, GAD-7 and BAARS-IV look the same. The validation
   and EN samples are close to symmetric but flat: median HiTOP skew .23 and .20, 10/83 items
   with |skew| > 1, median excess kurtosis ≈ −1.0. The preregistration's "non-normal" premise for
   the validation sample therefore holds as platykurtosis, not skew, and few validation items
   cross the Curran, West & Finch (1996) skew/kurtosis thresholds.
3. **The compared groups differ in floor.** The same HiTOP item draws a median ~28 percentage
   points more lowest-category responses in GP than in validation, and ~29 more than in EN;
   validation and EN are nearly identical (median −1). Fitting continuous multi-group models to
   Likert items whose floor differs between groups can produce spurious intercept and loading
   differences (Lubke & Muthén, 2004) — the specific risk for every ladder involving GP.
4. Taken together, the case for ordinal estimation rests on the four-category format (Rhemtulla,
   Brosseau-Liard & Savalei, 2012) and on these between-group differences in floor, more than
   on non-normality of the validation sample alone.